In [1]:
from pathlib import Path
import shutil
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

# Define the directories to be emptied
directories_to_clear = [
    PROJECT_ROOT / "working_files",
    PROJECT_ROOT / "extensions",
    PROJECT_ROOT / "experiments" / "snapshots"
]

# Function to delete all files and directories within a specified directory
def clear_directory(directory: Path):
    if directory.exists() and directory.is_dir():
        for item in directory.iterdir():
            try:
                if item.is_dir():
                    shutil.rmtree(item)  # Remove directory and all its contents
                else:
                    item.unlink()  # Remove file
            except Exception as e:
                print(f"Error deleting {item}: {e}")

# Clear all the directories
for directory in directories_to_clear:
    clear_directory(directory)

print("All specified directories have been cleared.")

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
All specified directories have been cleared.
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 74787e54-c92a-4dfc-8af8-a26f0862c066
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 0880d65a-b58b-493f-9e8f-894c15e17068
Seeded SystemPrompt 'format' with ID: 1 and GUID: bddf54eb-1b70-4b44-bc5b-02c008d65569
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 2049d159-91d6-44c2-be8d-87133055c874
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
✅ Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
✅ Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.enums.logging_enums import PROVIDER_TYPE
from app.enums.system_enums import SYSTEM
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/complex_lint_task.py")

# Construct input using session metadata
input_data = {
    "file_path":  str(file),
    "session_id": session_row.id,
    "system":     SYSTEM.LINTING.value
}

# Run associated program
from app.enums.logging_enums import RunContext, PROVIDER_TYPE

context = RunContext(
    called_by_type=PROVIDER_TYPE.SESSION,
    called_by_id=session_row.id,
    session_id=session_row.id,
    file_log_id=None,  # if available, otherwise use None
    execution_chain=[]
)

program = ProgramProviderFactory.create(
    id=session_row.program_provider_id,
    context=context
)

result = program.run(input_data, context=context)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))


✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing_controller",
  "state_type": "end",
  "decision": "accepted",
  "steps": 2,
  "max_steps": 20,
  "summary": "completed successfully",
  "output": {
    "state": "end",
    "file_path": "working_files\\complex_lint_task.py",
    "session_id": 1,
    "file_log_id": 1,
    "reason": "completed successfully",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing_controller",
    "decision": "accepted",
    "original_file": "tests\\complex_lint_task.py",
    "run_id": "71164b1e-6e1f-4dc4-b80f-1cb7594305d1",
    "transition_metadata": {
      "original_file": "tests\\complex_lint_task.py",
      "run_id": "71164b1e-6e1f-4dc4-b80f-1cb7594305d1",
      "agent_output": {},
      "file_path": null
    },
    "state_output": {
      "state": "end",
      "previous_state": "linting",
      "state_type": "end",
      "decision": "accepted",
      "steps": 3,
      "max_steps": 10,
  

In [3]:
# 🧾 Agent Conversation Log — Structured Trace View

import sqlite3
import pandas as pd
from datetime import datetime
from app.db.connection import DB_PATH

# 🗂️ Load from SQLite
conn = sqlite3.connect(DB_PATH)
df_convo = pd.read_sql("SELECT * FROM agent_conversation_log WHERE session_id='1'", conn)
conn.close()

# 📅 Normalize timestamps
df_convo["timestamp"] = pd.to_datetime(df_convo["timestamp"])
df_convo = df_convo.sort_values("timestamp").reset_index(drop=True)

# 🧠 Print conversation log entries
for i, row in df_convo.iterrows():
    print(f"\n🧩 Log {i + 1}")
    print(f"🕒 {row['timestamp']}")
    print(f"🧬 Run ID   : {row['run_id']}")
    print(f"📄 File Log : {row.get('file_log_id')}")
    print(f"🧠 Agent    : {row['agent_type']} (Provider ID {row['agent_provider_config_id']})")
    print("💬 Content  :")
    print(row["content"].strip())
    print("—" * 80)



🧩 Log 1
🕒 2025-06-16 23:42:48.049125+00:00
🧬 Run ID   : 95d537f9-39fd-477c-be87-34f0f1d470c1
📄 File Log : 1
🧠 Agent    : AGENT.GENERATOR (Provider ID 2)
💬 Content  :
- Added type hints to all functions to address `mypy` violations.
- Used `Optional` for the return type of `read_data_from_json` to indicate it can return `None`.
- Added exception handling in `read_data_from_json` to catch all exceptions, not just unspecified ones.
- Formatted the code using `black` to comply with PEP8 guidelines, addressing `black` violations.
- Ensured that `log_message` and other functions are called with typed arguments to resolve `mypy` errors.
- No significant tradeoffs were made; changes improve code clarity and maintainability.
————————————————————————————————————————————————————————————————————————————————

🧩 Log 2
🕒 2025-06-16 23:42:51.064159+00:00
🧬 Run ID   : 0d800ae2-eacd-4e6f-ba61-0adc0edf92c1
📄 File Log : 1
🧠 Agent    : AGENT.DISCRIMINATOR (Provider ID 3)
💬 Content  :
✅ Score improved from